In [13]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error,r2_score
import os
os.chdir('E:/Machine_Learning/Cases/Medical_Cost_Personal')

In [14]:
df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
X,y = df.drop('charges',axis=1),df['charges']

In [29]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')
X_ohe = ct.fit_transform(X)
X_ohe.head()

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age,bmi,children
0,0.0,1.0,0.0,0.0,1.0,19,27.900,0
1,1.0,0.0,0.0,1.0,0.0,18,33.770,1
2,1.0,0.0,0.0,1.0,0.0,28,33.000,3
3,1.0,0.0,1.0,0.0,0.0,33,22.705,0
4,1.0,0.0,1.0,0.0,0.0,32,28.880,0


### Linear Regression

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X_ohe,y,test_size=0.3,random_state=26)
lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)
r2_score(y_test,y_pred)

0.7625873682287878

### Ridge

In [23]:
alphas = np.linspace(0.001,10,20)
scores = []
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train,y_train)
    y_pred = ridge.predict(X_test)
    r2 = r2_score(y_test,y_pred)
    scores.append([a,r2])

In [24]:
df_r2_scores_ridge = pd.DataFrame(scores,columns=['alpha','r2 score'])
df_r2_scores_ridge.nlargest(1,'r2 score')

,alpha,r2 score
0,0.001,0.762587


### Lasso

In [25]:
alphas = np.linspace(0.001,10,20)
scores = []
for a in alphas:
    lasso = Lasso(alpha=a)
    lasso.fit(X_train,y_train)
    y_pred = lasso.predict(X_test)
    r2 = r2_score(y_test,y_pred)
    scores.append([a,r2])

df_r2_scores_lasso = pd.DataFrame(scores,columns=['alpha','r2 score'])
df_r2_scores_lasso.nlargest(1,'r2 score')

,alpha,r2 score
0,0.001,0.762587


### ElasticNet

In [26]:
alphas = np.linspace(0.001,10,20)
ratios = np.linspace(0.001,1,20)
scores = []
for a in alphas:
    for r in ratios:
        el = ElasticNet(alpha=a,l1_ratio=r)
        el.fit(X_train,y_train)
        y_pred = el.predict(X_test)
        r2 = r2_score(y_test,y_pred)
        scores.append([a,r,r2])

df_r2_scores_elastic = pd.DataFrame(scores,columns=['alpha','l1 ratio','r2 score'])
df_r2_scores_elastic.nlargest(1,'r2 score')

,alpha,l1 ratio,r2 score
19,0.001,1.0,0.762587


### All has the same r2 score so we choose LinearRegreesion as our best model

### We have testing dataset for this model in same folder "tst_insure"

In [27]:
tst = pd.read_csv('tst_insure.csv')
tst

,age,sex,bmi,children,smoker,region
0,19,female,27.900,0,yes,southwest
1,18,male,33.770,1,no,southeast
2,28,male,33.000,3,no,southeast
3,33,male,22.705,0,no,northwest
4,37,female,23.370,2,no,northwest
...,...,...,...,...,...,...
83,60,male,24.320,1,no,northwest
84,51,female,36.670,2,no,northwest
85,58,female,33.440,0,no,northwest
86,36,male,34.430,2,no,southeast


### Now we got our best model as Linear Regression : but the old "lr" is trained on 70% of data only , why not use 100% data when we have whole seperate testing dataset "tst_insure"  
### -- we just split data(70-30) to check which one is the better model .. but when we have different test dataset why not use our 100% data 

In [35]:
X , y = df.drop('charges',axis=1),df['charges']
X_ohe = ct.fit_transform(X)

In [38]:
lr = LinearRegression()
lr.fit(X_ohe,y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ -131.31,23848.53, -352.96,..., 256.86, 339.19, 475.5 ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['sex_male','smoker_yes','region_northwest',...,'age','bmi','children']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-1.194e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(8)


#### Also we have to transform() the testing data 'tst'

In [39]:
tst_ohe = ct.transform(tst)

In [40]:
lr.predict(tst_ohe)

array([25293.7130284 ,  3448.60283431,  6706.9884907 ,  3754.83016298,
        6090.13466947, 11806.85380404,   609.40593337,  2673.75489353,
       34189.57643384, 12710.99370832,  4503.41015199, 25429.39040984,
       13143.77592952, 26753.20814121,  9451.65140015, 12206.30871149,
       11205.52264619, 13842.84319194,  1125.54257025,  2962.27203677,
        2191.65550405,  3416.86353948,  1767.21811549, 13143.45521709,
        4963.21451834,  5218.98137096,  6104.30808337, 29009.37100989,
        6062.93582201, 30289.37010259,  9072.65703718, 35011.31592695,
       13399.38290374, 14493.39572269,  1958.96139904,  5227.3940846 ,
        9298.27550908, 10306.70717485, 30624.9137832 ,  7447.08139941,
       12283.78796336, 31355.86861593, 24941.29114668, 30500.89353988,
       10735.51245075, 31367.54473894, 27997.66702759, 14747.90041829,
        6379.77972163,  7134.56217392, 11474.74110792,  7166.19799263,
        8600.36139552,  6930.61334149, 17840.74651674,  3480.27001806,
      